# 03 — Progressives Fine-Tuning und Hard Negatives (Kreuzvalidierung)

Ersetzt die frueheren Notebooks 06, 07 und 08. Notebook 06 (eine Aufteilung,
36 Sturzfenster) ist durch Teil A vollstaendig abgedeckt.

| Teil | Frage |
|---|---|
| A | Lohnt der Zwischenschritt SisFall → KFall → Ziel? (5-fach, gepaart) |
| B | Hilft die Hoehergewichtung impactstarker Alltagsfenster? (5-fach, gepaart) |

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json, pickle, zipfile
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Ohne die
# aktuelle fall_lib waeren alle Ergebnisse falsch (andere Fensterlaenge).
import urllib.request
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    shutil.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import importlib, fall_lib as fl
importlib.reload(fl)
for _f in ("new_run", "build_windows", "znorm", "load_cappimu", "load_pilot",
           "augment_orientation", "rotation_between", "voting_eval", "finetune"):
    assert hasattr(fl, _f), f"fall_lib veraltet ({_f} fehlt) -- neue Version auf Drive legen."
assert fl.WIN_LEN == 100, "falsche Fensterlaenge -- fall_lib pruefen"
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"        # Basismodell (Teil 01)
PILOT = f"{BASE}/Pilotdaten"
CACHE = Path(f"{BASE}/cache"); CACHE.mkdir(exist_ok=True)


def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte
    (Drive ist fuer tausende Kleindateien zu langsam). Gibt None
    zurueck, wenn der Ordner fehlt."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel


def cached(name, fn):
    """Geladene Trials auf Drive zwischenspeichern (Laden dauert lange)."""
    p = CACHE / f"{name}.pkl"
    if p.exists():
        print(f"[cache] {name}")
        return pickle.load(open(p, "rb"))
    r = fn(); pickle.dump(r, open(p, "wb")); return r


from tensorflow import keras
def lade_modell(pfad):
    if not Path(pfad).exists():
        raise FileNotFoundError(f"{pfad} fehlt -- vorheriges Notebook ausfuehren.")
    m = keras.models.load_model(pfad, compile=False)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Konfiguration:", fl.default_config())

In [ ]:
basis = lade_modell(MODEL)
print(f"Basismodell: {basis.count_params():,} Parameter")

# Teil A — Progressives Fine-Tuning

In [ ]:
N_FOLDS = 5
ZIEL_POSITION = "ANKLE"

run = fl.new_run("07_progressiv_cv")

kf = fl.load_kfall(hole("KFALL/sensor_data_new", "/content/KFall"),
                   label_dir=hole("KFALL/label_data_new",
                                  "/content/KFall_labels"))
uma = fl.load_umafall(
    hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
         "/content/UMAFall"), position=ZIEL_POSITION)

X_kf, y_kf, _, _ = fl.build_windows(kf, mode="standard")
X_kf = fl.znorm(X_kf)

X, y, g, _ = fl.build_windows(uma, mode="standard")
X = fl.znorm(X)
print(f"\nZiel: {len(y)} Fenster, {int(y.sum())} Stuerze, "
      f"{len(np.unique(g))} Probanden")

## KFall-Zwischenstufe

Die KFall-Anpassung haengt nicht von der jeweiligen Falte ab, da die
KFall-Probanden ohnehin von den UMAFall-Probanden verschieden sind.
Sie wird daher einmal berechnet und in allen Falten wiederverwendet.

In [ ]:
print("Fine-Tuning auf KFall (einmalig) ...")
t = time.time()
m_kfall = fl.finetune(basis, X_kf, y_kf)
print(f"  fertig in {time.time()-t:.0f}s")

## Kreuzvalidierung

In [ ]:
splits = fl.group_kfold_splits(g, y, n_splits=N_FOLDS)

falten = {"A_zero_shot": [], "B_direkt": [],
          "C1_nach_kfall": [], "C2_progressiv": []}

for i, (tr, te) in enumerate(splits, 1):
    n_f = int(y[te].sum())
    print(f"\nFalte {i}/{N_FOLDS}: Test {len(te)} Fenster, "
          f"{n_f} Stuerze, {len(np.unique(g[te]))} Probanden")
    if n_f == 0:
        print("  keine Stuerze im Testsatz - Falte uebersprungen")
        continue

    falten["A_zero_shot"].append(
        fl.evaluate(basis, X[te], y[te], "  A Zero-Shot"))
    falten["C1_nach_kfall"].append(
        fl.evaluate(m_kfall, X[te], y[te], "  C1 nach KFall"))

    m_direkt = fl.finetune(basis, X[tr], y[tr])
    falten["B_direkt"].append(
        fl.evaluate(m_direkt, X[te], y[te], "  B Direkt"))

    m_prog = fl.finetune(m_kfall, X[tr], y[tr])
    falten["C2_progressiv"].append(
        fl.evaluate(m_prog, X[te], y[te], "  C2 Progressiv"))

## Zusammengefasstes Ergebnis

Aggregiert wird ueber die aufsummierten Konfusionsmatrizen, nicht
ueber die Mittelwerte der Falten -- sonst wuerden unterschiedlich
grosse Falten gleich stark gewichtet.

In [ ]:
gepoolt = {k: fl.pool_folds(v) for k, v in falten.items() if v}

print(f"Ueber {N_FOLDS} Falten zusammengefasst:\n")
namen = {"A_zero_shot": "A  Zero-Shot",
         "B_direkt": "B  Direkt",
         "C1_nach_kfall": "C1 nach KFall",
         "C2_progressiv": "C2 Progressiv"}
for k, n in namen.items():
    if k in gepoolt:
        fl.print_pooled(n, gepoolt[k])

print(f"\n{'Strategie':<16} {'Se je Falte'}")
print("-"*52)
for k, n in namen.items():
    if k in gepoolt:
        print(f"{n:<16} {gepoolt[k]['Se_per_fold']}")

## Gepaarter Vergleich

Entscheidend ist die Differenz **innerhalb** jeder Falte.

In [ ]:
vergleiche = {}
if falten["B_direkt"] and falten["C2_progressiv"]:
    print("Progressiv gegenueber direkt:\n")
    vergleiche["C2_vs_B"] = fl.paired_comparison(
        falten["B_direkt"], falten["C2_progressiv"],
        "Direkt", "Progressiv", metric="Se")
    print()
    vergleiche["C2_vs_B_Sp"] = fl.paired_comparison(
        falten["B_direkt"], falten["C2_progressiv"],
        "Direkt", "Progressiv", metric="Sp", verbose=True)

if falten["A_zero_shot"] and falten["B_direkt"]:
    print("\nFine-Tuning gegenueber Zero-Shot:\n")
    vergleiche["B_vs_A"] = fl.paired_comparison(
        falten["A_zero_shot"], falten["B_direkt"],
        "Zero-Shot", "Direkt", metric="Se")

## Einordnung

In [ ]:
v = vergleiche.get("C2_vs_B")
if v:
    print(f"Notebook 06 (eine Aufteilung): +5,6 pp, 36 Sturzfenster")
    print(f"Notebook 07 ({N_FOLDS} Falten):        "
          f"{v['mean_delta']*100:+.1f} pp, "
          f"{gepoolt['C2_progressiv']['TP'] + gepoolt['C2_progressiv']['FN']}"
          f" Sturzfenster")
    print()
    if v["significant"]:
        print("Der Unterschied ist ueber die Falten hinweg konsistent.")
    else:
        print("Der Unterschied ist statistisch nicht gesichert. Ein")
        print("Vorteil des progressiven Vorgehens laesst sich mit")
        print("diesen Daten nicht belegen; die geringe Zahl an")
        print("Sturzaufzeichnungen in UMAFall (157 insgesamt) duerfte")
        print("die Hauptursache sein.")

fl.save_results(run, {"gepoolt": gepoolt, "vergleiche": vergleiche,
                      "falten": falten, "n_folds": N_FOLDS,
                      "ziel_position": ZIEL_POSITION},
    config=fl.default_config(),
    notes=("Ersetzt Notebook 06. Fuenffache subjektunabhaengige "
           "Kreuzvalidierung mit gepaartem Vergleich."))

# Teil B — Hard-Negative-Training

In [ ]:
N_FOLDS     = 5
POSITION    = "ANKLE"
HARD_WEIGHT = 3.0     # Gewichtsfaktor fuer impactstarke ADL-Fenster
PEAK_THR    = 1.8     # Schwelle in g

run = fl.new_run("08_hardneg_cv")

uma = fl.load_umafall(
    hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
         "/content/UMAFall"), position=POSITION)

# Fensterung mit korrekter Beschriftung: innerhalb einer
# Sturzaufzeichnung gilt nur das Fenster als Sturz, das den
# Aufprall enthaelt.
X_raw, y, g, akt = fl.build_windows(uma, mode="sliding")
peaks = fl.acc_magnitude(X_raw).max(axis=1)   # vor der Normalisierung
X = fl.znorm(X_raw)

hart = (y == 0) & (peaks > PEAK_THR)
print(f"\n{len(y)} Fenster, {int(y.sum())} Stuerze")
print(f"Hard Negatives: {int(hart.sum())} von {int((y==0).sum())} "
      f"ADL-Fenstern ({100*hart.sum()/max((y==0).sum(),1):.1f}%)")

### Welche Aktivitaeten liefern die Hard Negatives?

Zur Kontrolle, ob die Auswahl tatsaechlich die erwarteten
Bewegungen mit Bodenkontakt trifft.

In [ ]:
print(f"{'Aktivitaet':<26} {'ADL-Fenster':>12} {'davon hart':>11} {'Anteil':>8}")
print("-"*60)
verteilung = {}
for a in sorted(np.unique(akt)):
    m = (akt == a) & (y == 0)
    if m.sum() == 0:
        continue
    h = int((hart & m).sum())
    print(f"{a:<26} {int(m.sum()):>12d} {h:>11d} "
          f"{100*h/m.sum():>7.1f}%")
    verteilung[a] = {"adl_windows": int(m.sum()), "hard": h,
                     "share": round(float(h/m.sum()), 4)}

## Kreuzvalidierung

In [ ]:
splits = fl.group_kfold_splits(g, y, n_splits=N_FOLDS)

# Gewichte: Klassenbalance mal Hard-Negative-Faktor
w_basis = np.ones(len(y), np.float32)
cw = fl.class_weights(y)
w_basis[y == 1] *= cw[1]
w_basis[y == 0] *= cw[0]
w_hart = w_basis.copy()
w_hart[hart] *= HARD_WEIGHT

falten = {"basis": [], "ohne_hard": [], "mit_hard": []}

for i, (tr, te) in enumerate(splits, 1):
    n_f = int(y[te].sum())
    print(f"\nFalte {i}/{N_FOLDS}: Test {len(te)} Fenster, "
          f"{n_f} Stuerze, {len(np.unique(g[te]))} Probanden")
    if n_f == 0:
        print("  keine Stuerze im Testsatz - uebersprungen")
        continue

    falten["basis"].append(
        fl.evaluate(basis, X[te], y[te], "  Basismodell"))

    m_ohne = fl.finetune(basis, X[tr], y[tr],
                         sample_weight=w_basis[tr])
    falten["ohne_hard"].append(
        fl.evaluate(m_ohne, X[te], y[te], "  FT ohne Hard Neg."))

    m_mit = fl.finetune(basis, X[tr], y[tr],
                        sample_weight=w_hart[tr])
    falten["mit_hard"].append(
        fl.evaluate(m_mit, X[te], y[te], "  FT mit Hard Neg."))

## Zusammengefasstes Ergebnis

In [ ]:
gepoolt = {k: fl.pool_folds(v) for k, v in falten.items() if v}

print(f"Ueber {N_FOLDS} Falten zusammengefasst:\n")
namen = {"basis": "Basismodell",
         "ohne_hard": "FT ohne Hard Neg.",
         "mit_hard": "FT mit Hard Neg."}
for k, n in namen.items():
    if k in gepoolt:
        fl.print_pooled(n, gepoolt[k])

print(f"\n{'Variante':<20} {'Se je Falte'}")
print("-"*54)
for k, n in namen.items():
    if k in gepoolt:
        print(f"{n:<20} {gepoolt[k]['Se_per_fold']}")

## Gepaarter Vergleich

Bewertet werden Sensitivitaet **und** Spezifitaet: Eine Massnahme,
die mehr Stuerze erkennt, aber ebenso viele zusaetzliche Fehlalarme
erzeugt, ist fuer den Einsatz in der Sohle kein Gewinn.

In [ ]:
vergleiche = {}
if falten["ohne_hard"] and falten["mit_hard"]:
    for metric in ["Se", "Sp"]:
        print()
        vergleiche[f"hard_vs_ohne_{metric}"] = fl.paired_comparison(
            falten["ohne_hard"], falten["mit_hard"],
            "ohne Hard Neg.", "mit Hard Neg.", metric=metric)

## Einordnung

In [ ]:
vs, vp = (vergleiche.get("hard_vs_ohne_Se"),
          vergleiche.get("hard_vs_ohne_Sp"))
if vs and vp:
    o, m = gepoolt["ohne_hard"], gepoolt["mit_hard"]
    print(f"{'':22} {'ohne':>10} {'mit':>10} {'Differenz':>12}")
    print("-"*58)
    print(f"{'Sensitivitaet':<22} {o['Se']:>10.3f} {m['Se']:>10.3f} "
          f"{vs['mean_delta']*100:>+11.1f}pp")
    print(f"{'Spezifitaet':<22} {o['Sp']:>10.3f} {m['Sp']:>10.3f} "
          f"{vp['mean_delta']*100:>+11.1f}pp")
    print(f"{'Fehlalarme (FP)':<22} {o['FP']:>10d} {m['FP']:>10d} "
          f"{m['FP']-o['FP']:>+12d}")
    print(f"{'Verpasste (FN)':<22} {o['FN']:>10d} {m['FN']:>10d} "
          f"{m['FN']-o['FN']:>+12d}")

    print()
    if vs["significant"] and vs["mean_delta"] > 0 and \
       not (vp["significant"] and vp["mean_delta"] < -0.02):
        print("Die Gewichtung verbessert die Sensitivitaet gesichert,")
        print("ohne die Spezifitaet nennenswert zu beeintraechtigen.")
        print("Der Ansatz ist fuer die Zielhardware zu empfehlen.")
    elif vs["significant"] and vp["significant"] and vp["mean_delta"] < 0:
        print("Die Gewichtung erhoeht die Sensitivitaet, senkt aber")
        print("zugleich gesichert die Spezifitaet. Ob dieser Tausch")
        print("lohnt, haengt davon ab, wie viele Fehlalarme im Alltag")
        print("hinnehmbar sind.")
    else:
        print("Kein gesicherter Effekt. Die gegenlaeufigen Ergebnisse")
        print("frueherer Einzelaufteilungen sind demnach auf")
        print("Zufallsschwankung zurueckzufuehren; der Ansatz wird")
        print("nicht weiterverfolgt.")

fl.save_results(run,
    {"gepoolt": gepoolt, "vergleiche": vergleiche, "falten": falten,
     "config": {"hard_weight": HARD_WEIGHT, "peak_g": PEAK_THR,
                "n_hard": int(hart.sum()), "n_folds": N_FOLDS},
     "hard_negatives_nach_aktivitaet": verteilung},
    config=fl.default_config(),
    notes=("Ersetzt den Hard-Negative-Teil von Notebook 03. "
           "Fuenffache Kreuzvalidierung, gepaarter Vergleich fuer "
           "Sensitivitaet und Spezifitaet."))